# Motor-insurance claims: high-value claim detection (XGBoost)

The data (`BASE.csv`) is confidential and is **not** included in this repository. Outputs were cleared before publishing. See `README.md` for the approach, results and limitations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling  import SMOTE
from sklearn.linear_model    import LogisticRegression
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import LinearSVC
from sklearn.calibration     import CalibratedClassifierCV
from sklearn.ensemble        import RandomForestClassifier, IsolationForest
from xgboost                 import XGBClassifier
from sklearn.metrics         import (classification_report, roc_auc_score,
                                     roc_curve, ConfusionMatrixDisplay,
                                     precision_score, recall_score, f1_score)

df = pd.read_csv("BASE.csv", sep=";", encoding="latin-1", low_memory=False)
df.columns = df.columns.str.strip()

print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
display(df.head())

# Distribution variable cible
print("\n=== VARIABLE CIBLE : NV CID ===")
print(df['NV CID'].value_counts())
print(df['NV CID'].value_counts(normalize=True).mul(100).round(2))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['NV CID'].value_counts()

axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#2196F3','#F44336'], startangle=90)
axes[0].set_title("Répartition NV CID", fontweight='bold')

axes[1].bar(counts.index, counts.values,
            color=['#2196F3','#F44336'], edgecolor='white', width=0.5)
axes[1].set_title("Nombre de sinistres par classe", fontweight='bold')
axes[1].set_ylabel("Nombre")
for i, v in enumerate(counts.values):
    axes[1].text(i, v+100, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
df_ml = df.copy()
df_ml.columns = df_ml.columns.str.strip()

# 1. Suppression colonnes inutiles + leakage
cols_suppr = [
    'Code Cie',           # vide
    'Cas de Barème',      # vide
    'Cie Adv',            # vide
    'Nom Assuré',         # identifiant
    'Nom Advesaire',      # identifiant
    'N° Sinistre Cie',    # identifiant
    'N° Police Advers',   # identifiant
    'Imm Advers',         # identifiant
    'BORDEREAU',          # administratif
    'Montant Réel',       # leakage direct avec NV CID
    'Montant Recours',    # leakage direct avec NV CID
    'Quintile',           # basé sur le montant → leakage
]
df_ml.drop(columns=[c for c in cols_suppr if c in df_ml.columns], inplace=True)
print("✓ Colonnes supprimées")

# 2. Gestion des valeurs manquantes
print("\n=== VALEURS MANQUANTES AVANT TRAITEMENT ===")
missing = df_ml.isnull().sum()
missing_pct = (missing / len(df_ml) * 100).round(2)
df_missing = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct
})
display(df_missing[df_missing['Valeurs manquantes'] > 0])

# Colonnes numériques → médiane
for col in df_ml.select_dtypes(include=[np.number]).columns:
    if df_ml[col].isnull().sum() > 0:
        df_ml[col].fillna(df_ml[col].median(), inplace=True)

# Colonnes catégorielles → mode
for col in df_ml.select_dtypes(include=['object']).columns:
    if df_ml[col].isnull().sum() > 0:
        df_ml[col].fillna(df_ml[col].mode()[0], inplace=True)

print("\n=== VALEURS MANQUANTES APRÈS TRAITEMENT ===")
print(df_ml.isnull().sum().sum(), "valeurs manquantes restantes")
print("✓ Gestion des valeurs manquantes terminée")

# 3. Nettoyage Mois Bordereau
mois_map = {
    'janv':1,'févr':2,'mars':3,'avr':4,'mai':5,'juin':6,
    'juil':7,'août':8,'sept':9,'oct':10,'nov':11,'déc':12
}
df_ml['mois_bordereau'] = (df_ml['Mois Bordereau']
                           .astype(str).str.strip()
                           .str.split('-').str[0].str.strip()
                           .map(mois_map))
df_ml['mois_bordereau'].fillna(0, inplace=True)
df_ml.drop(columns=['Mois Bordereau'], inplace=True)
print("✓ Mois Bordereau → numérique")

# 4. Parsing Date Sinistre
df_ml['Date Sinistre'] = pd.to_datetime(
    df_ml['Date Sinistre'], dayfirst=True, errors='coerce')
df_ml['mois_sinistre']  = df_ml['Date Sinistre'].dt.month
df_ml['annee_sinistre'] = df_ml['Date Sinistre'].dt.year
df_ml['jour_semaine']   = df_ml['Date Sinistre'].dt.dayofweek
df_ml['weekend']        = (df_ml['jour_semaine'] >= 5).astype(int)
df_ml.drop(columns=['Date Sinistre'], inplace=True)
print("✓ Date → mois / année / jour_semaine / weekend")

# 5. Nettoyage Usage
df_ml['Usage'] = df_ml['Usage'].astype(str).str.strip().str.upper()
usage_map = {'E':1,'A':2,'B':3,'C1':4,'C2':5,'D':6}
df_ml['Usage'] = df_ml['Usage'].map(usage_map).fillna(0)
print("✓ Usage → numérique")

# 6. Encodage variables catégorielles
le = LabelEncoder()
for col in ['Nom Compagnie','CIE SORTANT','Nom Compagnie Adverse',
            'CID ENTRANT','Type de recours']:
    if col in df_ml.columns:
        df_ml[col] = le.fit_transform(df_ml[col].astype(str))
print("✓ Encodage LabelEncoder appliqué")

# 7. Feature Engineering
df_ml['debut_semaine']   = (df_ml['jour_semaine'] <= 1).astype(int)
df_ml['sinistre_recent'] = (df_ml['annee_sinistre'] >= 2024).astype(int)
df_ml['haute_saison']    = (df_ml['mois_sinistre'].isin([6, 7, 8])).astype(int)
df_ml['resp_partagee']   = (df_ml['Tx Resp.'] == 50).astype(int)
df_ml['rma_vs_adverse']  = (df_ml['Nom Compagnie'] == df_ml['Nom Compagnie Adverse']).astype(int)

print("✓ Feature Engineering — 5 nouvelles variables créées :")
print("  - debut_semaine    : sinistre déclaré lundi ou mardi")
print("  - sinistre_recent  : sinistre de 2024 ou 2025")
print("  - haute_saison     : sinistre en juin, juillet ou août")
print("  - resp_partagee    : taux de responsabilité = 50%")
print("  - rma_vs_adverse   : même compagnie des deux côtés")

# 8. Variable cible
df_ml['fraude'] = (df_ml['NV CID'].str.strip() == 'SUP 10000').astype(int)
df_ml.drop(columns=['NV CID'], inplace=True)

print("\n=== DISTRIBUTION CIBLE ===")
print(df_ml['fraude'].value_counts())
print(df_ml['fraude'].value_counts(normalize=True).mul(100).round(2))

# 9. Séparation X / y  (SANS normalisation ici — appliquée après le split)
X = df_ml.drop(columns=['fraude']).select_dtypes(include=[np.number])
y = df_ml['fraude']

print(f"\n✓ Features retenues ({X.shape[1]}) :")
print(list(X.columns))

In [ ]:
# ── Split stratifié 80/20 ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")
print(f"\nDistribution Train avant SMOTE :\n{y_train.value_counts()}")

# ── SMOTE uniquement sur le train ──────────────────────────────────────────
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"\n✓ Après SMOTE :")
print(pd.Series(y_train_res).value_counts())

# Normalisation APRÈS le split (évite le data leakage)
# Le scaler est fit UNIQUEMENT sur les données d'entraînement,
# puis appliqué (transform) sur train et test séparément.
scaler = StandardScaler()
X_train_res = pd.DataFrame(
    scaler.fit_transform(X_train_res),   # fit + transform sur train
    columns=X.columns
)
X_test = pd.DataFrame(
    scaler.transform(X_test),            # transform seulement sur test
    columns=X.columns
)
print("\n✓ StandardScaler appliqué correctement (fit sur train, transform sur test)")
display(X_train_res.head())

In [ ]:
ratio = (y_train_res==0).sum() / (y_train_res==1).sum()

modeles = {
    "Régression Logistique": LogisticRegression(
        C=1.0, solver='lbfgs', class_weight='balanced',
        max_iter=1000, random_state=42),

    "KNN": KNeighborsClassifier(
        n_neighbors=7, weights='distance', n_jobs=-1),

    "SVM (Linéaire)": CalibratedClassifierCV(
        LinearSVC(C=1.0, class_weight='balanced',
                  max_iter=2000, random_state=42),
        cv=3),

    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=8, class_weight='balanced',
        random_state=42, n_jobs=-1),

    "XGBoost": XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=6,
        scale_pos_weight=ratio, eval_metric='logloss',
        random_state=42, n_jobs=-1),
}

print("=== ENTRAÎNEMENT ===")
for nom, modele in modeles.items():
    print(f"⏳ {nom} ...", end=' ', flush=True)
    scores = cross_val_score(modele, X_train_res, y_train_res,
                             cv=3, scoring='roc_auc', n_jobs=1)
    modele.fit(X_train_res, y_train_res)
    print(f"✓  AUC-CV : {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
print("=== ÉVALUATION SUR TEST ===")
resultats = {}

for nom, modele in modeles.items():
    y_pred  = modele.predict(X_test)
    y_proba = modele.predict_proba(X_test)[:, 1]

    resultats[nom] = {
        "Precision" : precision_score(y_test, y_pred, zero_division=0),
        "Recall"    : recall_score(y_test, y_pred, zero_division=0),
        "F1-Score"  : f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC"   : roc_auc_score(y_test, y_proba),
    }

    print(f"\n{'='*50}\n {nom}\n{'='*50}")
    print(classification_report(y_test, y_pred,
          target_names=['INF 10000','SUP 10000'], zero_division=0))

    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['INF 10000','SUP 10000'],
        cmap='Blues', ax=ax)
    ax.set_title(f"Matrice de confusion — {nom}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Isolation Forest ──────────────────────────────────────────────────────
# Entraîné sur X_train_res (données d'entraînement normalisées).
# Note : l'Isolation Forest étant un modèle non supervisé de détection
# d'anomalies, il est entraîné sur les données d'entraînement disponibles.
iso = IsolationForest(
    n_estimators=100,
    contamination=float(y.mean()),   # taux de fraude réel de la base
    random_state=42
)
iso.fit(X_train_res)

y_pred_iso = np.where(iso.predict(X_test) == -1, 1, 0)
scores_iso = iso.decision_function(X_test)

print("=== ISOLATION FOREST ===")
print(classification_report(y_test, y_pred_iso,
      target_names=['INF 10000','SUP 10000'], zero_division=0))

# ── Visualisation ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogramme scores anomalie
axes[0].hist(scores_iso[y_test.values==0], bins=50, alpha=0.6,
             color='#2196F3', label='INF 10000')
axes[0].hist(scores_iso[y_test.values==1], bins=50, alpha=0.7,
             color='#F44336', label='SUP 10000')
axes[0].axvline(0, color='black', linestyle='--', label='Seuil')
axes[0].set_title("Distribution scores anomalie", fontweight='bold')
axes[0].set_xlabel("Score anomalie")
axes[0].legend()

# Boxplot
df_iso_viz = pd.DataFrame({
    'score_anomalie' : scores_iso,
    'fraude_reelle'  : y_test.values
})
df_iso_viz['classe'] = df_iso_viz['fraude_reelle'].map(
    {0: 'INF 10000', 1: 'SUP 10000'})

sns.boxplot(data=df_iso_viz, x='classe', y='score_anomalie',
            palette=['#2196F3','#F44336'], ax=axes[1])
axes[1].set_title("Boxplot scores — Isolation Forest", fontweight='bold')
axes[1].set_xlabel("Classe réelle")
axes[1].set_ylabel("Score anomalie")

plt.tight_layout()
plt.show()

In [ ]:
# ── Tableau comparatif ────────────────────────────────────────────────────
df_res = pd.DataFrame(resultats).T.sort_values("ROC-AUC", ascending=False)
df_res.loc["Isolation Forest"] = {
    "Precision" : precision_score(y_test, y_pred_iso, zero_division=0),
    "Recall"    : recall_score(y_test, y_pred_iso, zero_division=0),
    "F1-Score"  : f1_score(y_test, y_pred_iso, zero_division=0),
    "ROC-AUC"   : np.nan,
}
print("=== TABLEAU COMPARATIF FINAL ===")
display(df_res.round(4))

# ── Courbe ROC ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
couleurs = ['#1565C0','#2E7D32','#F57F17','#B71C1C','#6A1B9A']

for (nom, modele), couleur in zip(modeles.items(), couleurs):
    y_proba = modele.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr,
            label=f"{nom} (AUC={auc:.3f})",
            lw=2.5 if nom=="XGBoost" else 1.5,
            color=couleur)

ax.plot([0,1],[0,1], 'k--', lw=1, label='Aléatoire')
ax.set_xlabel("Taux Faux Positifs (FPR)")
ax.set_ylabel("Taux Vrais Positifs (TPR)")
ax.set_title("Courbe ROC — Comparaison des modèles", fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fi_df = (pd.DataFrame({
    'Feature'    : X.columns,
    'Importance' : modeles["XGBoost"].feature_importances_})
    .sort_values('Importance', ascending=False))

colors = ['#F44336' if i < 3 else '#FF9800' if i < 7 else '#2196F3'
          for i in range(len(fi_df))]

plt.figure(figsize=(9, 5))
bars = plt.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1],
                color=colors[::-1], edgecolor='white')
for bar, val in zip(bars, fi_df['Importance'][::-1]):
    plt.text(bar.get_width()+0.001,
             bar.get_y()+bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9)

plt.title("Feature Importance — XGBoost", fontweight='bold')
plt.xlabel("Importance (gain)")
plt.tight_layout()
plt.show()

display(fi_df.reset_index(drop=True))

In [ ]:
# Calcul du Fraud Score 
fraud_scores = modeles["XGBoost"].predict_proba(X_test)[:, 1]

df_scores = pd.DataFrame({
    'fraud_score'   : fraud_scores,
    'classe_reelle' : y_test.values,
    'prediction'    : modeles["XGBoost"].predict(X_test),
}).sort_values('fraud_score', ascending=False)

df_scores['niveau_risque'] = pd.cut(
    df_scores['fraud_score'],
    bins=[0, 0.30, 0.70, 1.0],
    labels=['Faible','Modéré','Élevé']
)

print("=== TOP 20 SINISTRES LES PLUS SUSPECTS ===")
display(df_scores.head(20))

print("\n=== RÉPARTITION NIVEAUX DE RISQUE ===")
print(df_scores['niveau_risque'].value_counts())

#  Visualisation 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df_scores[df_scores['classe_reelle']==0]['fraud_score'],
             bins=50, alpha=0.6, color='#2196F3', label='INF 10000')
axes[0].hist(df_scores[df_scores['classe_reelle']==1]['fraud_score'],
             bins=50, alpha=0.7, color='#F44336', label='SUP 10000')
axes[0].axvline(0.30, color='orange', linestyle='--', lw=1.5, label='Seuil 0.30')
axes[0].axvline(0.70, color='red',    linestyle='--', lw=1.5, label='Seuil 0.70')
axes[0].set_title("Distribution du Fraud Score", fontweight='bold')
axes[0].set_xlabel("Fraud Score")
axes[0].legend()

counts_risque = df_scores['niveau_risque'].value_counts()
axes[1].bar(counts_risque.index, counts_risque.values,
            color=['#4CAF50','#FF9800','#F44336'], edgecolor='white')
axes[1].set_title("Répartition par niveau de risque", fontweight='bold')
axes[1].set_ylabel("Nombre de sinistres")
for i, v in enumerate(counts_risque.values):
    axes[1].text(i, v+50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Export 
df_export = df_ml.loc[y_test.index].copy()
df_export['fraud_score'] = fraud_scores
df_export['prediction'] = modeles["XGBoost"].predict(X_test)
df_export['niveau_risque'] = pd.cut(
    df_export['fraud_score'],
    bins=[0, 0.30, 0.70, 1.0],
    labels=['Faible', 'Modéré', 'Élevé']
)
df_export.to_csv("fraud_scores_RMA_final.csv", index=False)
print("✓ Export terminé !")
print(f"Colonnes : {list(df_export.columns)}")